In [26]:
import polars as pl
import numpy as np
from pathlib import Path
from project_paths import paths
from imd_features.config import FeatureSetConfig, GroupConfig, ReductionMethod
from imd_features.process import create_feature_set
from imd_features.evaluate import evaluate
from imd_features.diagnostic import group_summary

In [27]:
input_df = pl.read_parquet(paths.input_file)
input_df.shape

(268, 439)

In [28]:
domain_proxy = FeatureSetConfig(
    name="domain_proxy",
    description="Cross-source features grouped by IMD domain, reduced to domain-level factors",
    groups={
        "income_employment": GroupConfig(
            columns=[
                "total_claims", 
                "mean_monthly_claims",
                "total_nwr_claims", 
                "total_planfw_claims",
                "total_prepfw_claims", 
                "total_sfw_claims",
                "%_claims_nwr", 
                "%_claims_planfw",
                "%_claims_prepfw", 
                "%_claims_sfw",
            ],
            scale=True,
            reduction_method=ReductionMethod.FA,
            n_components=2,
        ),
        "crime": GroupConfig(
            columns=[
                "violent-crime", 
                "burglary", 
                "anti-social-behaviour",
                "shoplifting", 
                "criminal-damage-arson", 
                "drugs",
                "robbery", 
                "vehicle-crime", 
                "other-theft",
                "public-order", 
                "other-crime", 
                "theft-from-the-person",
                "total_crimes", 
                "resolution_rate",
            ],
            scale=True,
            reduction_method=ReductionMethod.FA,
            n_components=3,
        ),
        "barriers_housing": GroupConfig(
            columns=[
                "Overall (walking)", 
                "Overall (cycling)",
                "Overall (public transport)", 
                "Overall (driving)", 
                "Overall",
                "Shopping (walking)", 
                "Shopping (public transport)",
                "Residential (walking)", 
                "Residential (public transport)",
                "lsoa_mean_price", 
                "lsoa_max_price",
                "T_mean_price", 
                "F_mean_price", 
                "S_mean_price",
                "D_mean_price", 
                "O_mean_price",
                "total_transactions",
                "nearest_shop",
                "count_essential_services_500", 
                "count_essential_services_1000",
            ],
            scale=True,
            reduction_method=ReductionMethod.FA,
            n_components=5,
        ),
        "living_environment": GroupConfig(
            columns=[
                "landuse_residential_0", 
                "landuse_commercial_0",
                "landuse_industrial_0", 
                "landuse_grass_0",
                "landuse_recreation_ground_0", 
                "landuse_education_0",
                "streetlit_percentage",
                "count_fast_food_takeaway_1000",
                "count_alcohol_gambling_1000",
                "ratio_fast_food_takeaway_to_food_dining_1000",
                "count_food_dining_1000",
            ],
            scale=True,
            reduction_method=ReductionMethod.FA,
            n_components=3,
        ),
        "education_health": GroupConfig(
            columns=[
                "Education (walking)", 
                "Education (cycling)",
                "Education (public transport)", 
                "Education (driving)",
                "Education (overall)",
                "Healthcare (walking)", 
                "Healthcare (cycling)",
                "Healthcare (public transport)", 
                "Healthcare (driving)",
                "Healthcare (overall)",
                "count_education_skills_500", 
                "count_education_skills_1000",
                "count_healthcare_access_500", 
                "count_healthcare_access_1000",
                "count_childcare_early_years_1000",
                "count_higher_education_research_1000",
            ],
            scale=True,
            reduction_method=ReductionMethod.FA,
            n_components=4,
        ),
        "demographics": GroupConfig(
            columns=[
                "lsoa_population", 
                "aged_under_15",
                "working_age_population", 
                "pension_age_population",
            ],
            scale=True,
        ),
    },
)

In [29]:
# ~28 features testing how efficient we can get selecting features
compact_30 = FeatureSetConfig(
    name="compact_30",
    description="~28 features chosen based on intuitively trying to get as much information in as few raw features",
    groups={
        "economic": GroupConfig(
            columns=[
                "total_claims", 
                "%_claims_nwr",
                "lsoa_mean_price", 
                "total_transactions",
                "working_age_population",
            ],
            scale=True,
        ),
        "crime": GroupConfig(
            columns=[
                "total_crimes", 
                "violent-crime",
                "anti-social-behaviour", 
                "resolution_rate",
            ],
            scale=True,
        ),
        "access": GroupConfig(
            columns=[
                "Overall", 
                "Overall (walking)", 
                "Overall (public transport)",
                "Healthcare (walking)", 
                "nearest_shop",
            ],
            scale=True,
        ),
        "environment": GroupConfig(
            columns=[
                "landuse_residential_0", 
                "landuse_industrial_0",
                "streetlit_percentage", 
                "ratio_fast_food_takeaway_to_food_dining_1000",
                "landuse_grass_0",
            ],
            scale=True,
        ),
        "services_1000": GroupConfig(
            columns=[
                "count_healthcare_access_1000", 
                "count_education_skills_1000",
                "count_essential_services_1000", 
                "count_transport_public_1000",
                "count_financial_services_1000", 
                "count_retail_commerce_1000",
            ],
            scale=True,
        ),
        "demographics": GroupConfig(
            columns=["lsoa_population", "aged_under_15", "pension_age_population"],
            scale=True,
        ),
    },
)

In [ ]:
# basically a better thought through version of the mixed example 
# connectivity factor analysis with 8 factors
# OSM PCA with 8 components
# all uc features are selected and in crime the ones that have really low occurance across lsoas are excluded
calibrated_reduction = FeatureSetConfig(
    name="calibrated_reduction",
    description="calibrated reduction: connectivity FA(8), curated OSM PCA(8)",
    groups={
        "crime": GroupConfig(
            columns=[
                "violent-crime", 
                "burglary", 
                "anti-social-behaviour",
                "shoplifting", 
                "criminal-damage-arson", 
                "drugs",
                "robbery", 
                "vehicle-crime", 
                "total_crimes", 
                "resolution_rate",
            ],
            scale=True,
        ),
        "uc": GroupConfig(
            columns=[
                "total_claims", 
                "mean_monthly_claims",
                "%_claims_nwr", 
                "%_claims_planfw", 
                "%_claims_prepfw", 
                "%_claims_sfw",
            ],
            scale=True,
        ),
        "connectivity": GroupConfig(
            columns=[
                "Employment (walking)", 
                "Education (walking)",
                "Healthcare (walking)", 
                "Leisure and Community (walking)",
                "Shopping (walking)", 
                "Residential (walking)", 
                "Overall (walking)",
                "Employment (cycling)", 
                "Education (cycling)",
                "Healthcare (cycling)", 
                "Leisure and Community (cycling)",
                "Shopping (cycling)", 
                "Residential (cycling)", 
                "Overall (cycling)",
                "Business (public transport)", 
                "Education (public transport)",
                "Healthcare (public transport)", 
                "Leisure and Community (public transport)",
                "Shopping (public transport)", 
                "Residential (public transport)",
                "Overall (public transport)",
                "Employment (driving)", 
                "Education (driving)",
                "Healthcare (driving)", 
                "Leisure and Community (driving)",
                "Shopping (driving)", 
                "Residential (driving)", 
                "Overall (driving)",
                "Employment (overall)", 
                "Education (overall)",
                "Healthcare (overall)", 
                "Leisure and Community (overall)",
                "Shopping (overall)", 
                "Residential (overall)", 
                "Overall",
            ],
            scale=True,
            reduction_method=ReductionMethod.FA,
            n_components=8,
        ),
        "land_registry": GroupConfig(
            columns=[
                "lsoa_mean_price", 
                "lsoa_max_price",
                "T_mean_price", 
                "F_mean_price", 
                "S_mean_price", 
                "D_mean_price",
                "total_transactions",
            ],
            scale=True,
        ),
        "osm_amenities": GroupConfig(
            columns=[
                f"count_{group}_{dist}"
                for group in [
                    "healthcare_access", 
                    "education_skills", 
                    "financial_services",
                    "food_dining", 
                    "fast_food_takeaway", 
                    "alcohol_gambling",
                    "transport_public", 
                    "essential_services", 
                    "childcare_early_years",
                    "retail_commerce", 
                    "community_social", 
                    "social_support",
                ]
                for dist in [500, 1000]
            ] + ["nearest_shop", "ratio_fast_food_takeaway_to_food_dining_1000"],
            scale=True,
            reduction_method=ReductionMethod.PCA,
            n_components=8,
        ),
        "landuse_environment": GroupConfig(
            columns=[
                "landuse_residential_0", 
                "landuse_commercial_0",
                "landuse_industrial_0", 
                "landuse_grass_0",
                "landuse_recreation_ground_0", 
                "landuse_education_0",
                "streetlit_percentage",
            ],
            scale=True,
        ),
        "population": GroupConfig(
            columns=[
                "lsoa_population", 
                "aged_under_15",
                "working_age_population", 
                "pension_age_population",
            ],
            scale=True,
        ),
    },
)

In [31]:
# OSM amenities at 3 meaningful scales (walking distance, short drive, trip)
DEPRIVATION_AMENITY_GROUPS = [
    "healthcare_access", 
    "education_skills", 
    "food_dining",
    "fast_food_takeaway", 
    "essential_services", 
    "transport_public",
    "financial_services", 
    "childcare_early_years", 
    "retail_commerce",
    "social_support", 
    "community_social",
]

multiscale_access = FeatureSetConfig(
    name="multiscale_access",
    description="OSM amenities at 3 meaningful scales (500, 1000, 2500m) + all sources",
    groups={
        "walkable_500": GroupConfig(
            columns=[f"count_{g}_500" for g in DEPRIVATION_AMENITY_GROUPS],
            scale=True,
            reduction_method=ReductionMethod.FA,
            n_components=4
        ),
        "neighbourhood_1000": GroupConfig(
            columns=[f"count_{g}_1000" for g in DEPRIVATION_AMENITY_GROUPS],
            scale=True,
            reduction_method=ReductionMethod.FA,
            n_components=4
        ),
        "district_2500": GroupConfig(
            columns=[f"count_{g}_2500" for g in DEPRIVATION_AMENITY_GROUPS],
            scale=True,
            reduction_method=ReductionMethod.FA,
            n_components=4
        ),
        "osm_extra": GroupConfig(
            columns=["nearest_shop", "ratio_fast_food_takeaway_to_food_dining_1000", "streetlit_percentage"],
            scale=True,
            reduction_method=ReductionMethod.FA,
            n_components=4
        ),
        "land_registry": GroupConfig(
            columns=[
                "lsoa_max_price",
                "T_mean_price", 
                "F_mean_price", 
                "S_mean_price", 
                "D_mean_price", 
                "O_mean_price",
                "T_count_transactions", 
                "F_count_transactions",
                "S_count_transactions", 
                "D_count_transactions", 
                "O_count_transactions",
            ],
            scale=True,
            reduction_method=ReductionMethod.FA,
            n_components=4,
        ),
        "economic": GroupConfig(
            columns=[
                "total_claims", 
                "mean_monthly_claims",
                "%_claims_nwr", 
                "%_claims_planfw", 
                "%_claims_prepfw", 
                "%_claims_sfw",
                "lsoa_mean_price", 
                "total_transactions",
            ],
            scale=True,
        ),
        "crime": GroupConfig(
            columns=[
                "violent-crime", 
                "burglary", 
                "anti-social-behaviour",
                "shoplifting", 
                "criminal-damage-arson", 
                "drugs",
                "total_crimes", 
                "resolution_rate",
            ],
            scale=True,
        ),
        "connectivity_summary": GroupConfig(
            columns=[
                "Overall (walking)", 
                "Overall (cycling)",
                "Overall (public transport)", 
                "Overall (driving)", 
                "Overall",
            ],
            scale=True,
        ),
        "landuse": GroupConfig(
            columns=[
                "landuse_residential_0", 
                "landuse_commercial_0",
                "landuse_industrial_0", 
                "landuse_grass_0",
                "landuse_recreation_ground_0", 
                "landuse_education_0",
            ],
            scale=True,
        ),
        "population": GroupConfig(
            columns=[
                "lsoa_population", 
                "aged_under_15",
                "working_age_population", 
                "pension_age_population",
            ],
            scale=True,
        ),
    },
)

In [32]:
# Create population rate features
raw = pl.read_parquet(paths.input_file)
engineered = raw.with_columns(
    (pl.col("total_crimes") / pl.col("lsoa_population") * 1000).alias("crime_rate_per_1000"),
    (pl.col("violent-crime") / pl.col("lsoa_population") * 1000).alias("violent_crime_rate"),
    (pl.col("anti-social-behaviour") / pl.col("lsoa_population") * 1000).alias("asb_rate"),
    (pl.col("burglary") / pl.col("lsoa_population") * 1000).alias("burglary_rate"),
    (pl.col("drugs") / pl.col("lsoa_population") * 1000).alias("drugs_rate"),
    (pl.col("total_claims") / pl.col("working_age_population")).alias("uc_claim_rate"),
    (pl.col("total_nwr_claims") / pl.col("working_age_population")).alias("uc_nwr_rate"),
    (pl.col("aged_under_15") / pl.col("lsoa_population")).alias("youth_share"),
    (pl.col("pension_age_population") / pl.col("lsoa_population")).alias("elderly_share"),
    (pl.col("total_transactions") / pl.col("lsoa_population") * 1000).alias("transactions_per_capita"),
)

engineered_path = paths.input_file.parent / "combined_engineered.parquet"
engineered.write_parquet(engineered_path)
del raw, engineered

In [33]:
# population rates instead of raw counts
engineered_rates = FeatureSetConfig(
    name="engineered_rates",
    description="Population crime/UC rates testing whether rates outperform counts",
    groups={
        "crime_rates": GroupConfig(
            columns=[
                "crime_rate_per_1000", 
                "violent_crime_rate",
                "asb_rate", 
                "burglary_rate", 
                "drugs_rate",
                "resolution_rate",
            ],
            scale=True,
        ),
        "benefit_rates": GroupConfig(
            columns=[
                "uc_claim_rate", 
                "uc_nwr_rate",
                "%_claims_planfw", 
                "%_claims_sfw",
            ],
            scale=True,
        ),
        "access": GroupConfig(
            columns=[
                "Overall", 
                "Overall (walking)", 
                "Overall (public transport)",
                "Healthcare (walking)", 
                "nearest_shop",
            ],
            scale=True,
        ),
        "housing": GroupConfig(
            columns=[
                "lsoa_mean_price", 
                "lsoa_max_price",
                "total_transactions", 
                "transactions_per_capita",
            ],
            scale=True,
        ),
        "services_1000": GroupConfig(
            columns=[
                "count_healthcare_access_1000", 
                "count_education_skills_1000",
                "count_essential_services_1000", 
                "count_transport_public_1000",
                "count_financial_services_1000", 
                "count_retail_commerce_1000",
                "count_fast_food_takeaway_1000", 
                "count_food_dining_1000",
            ],
            scale=True,
        ),
        "environment": GroupConfig(
            columns=[
                "landuse_residential_0", 
                "landuse_industrial_0",
                "streetlit_percentage", 
                "ratio_fast_food_takeaway_to_food_dining_1000",
                "landuse_grass_0", 
                "landuse_commercial_0",
            ],
            scale=True,
        ),
        "demographics": GroupConfig(
            columns=[
                "lsoa_population", 
                "youth_share",
                "elderly_share", 
                "working_age_population",
            ],
            scale=True,
        ),
    },
)

In [ ]:
configs = {
    "domain_proxy": (domain_proxy, paths.input_file),
    "compact_30": (compact_30, paths.input_file),
    "calibrated_reduction": (calibrated_reduction, paths.input_file),
    "multiscale_access": (multiscale_access, paths.input_file),
    "engineered_rates": (engineered_rates, engineered_path),
}

all_results = {}
all_metadata = {}

for name, (config, input_path) in configs.items():
    df, metadata, _ = create_feature_set(input_path, config)
    results = evaluate(df, config)
    n_features = df.shape[1] - 1
    all_results[name] = {"n_features": n_features, **results}
    all_metadata[name] = (config, metadata)

In [35]:
configs_created = {
    "domain_proxy": domain_proxy,
    "compact_30": compact_30,
    "calibrated_reduction": calibrated_reduction,
    "multiscale_access": multiscale_access,
    "engineered_rates": engineered_rates,
}

results = {}
for name, config in configs_created.items():
    df = pl.read_parquet(paths.output / f"{config.output_name}.parquet")
    results[name] = evaluate(df, config)

In [36]:
rows = []
for name, result in results.items():
    for model_name, metrics in result.items():
        rows.append({
            "config": name,
            "model": model_name,
            "r2_mean": metrics["r2_mean"],
            "rmse_mean": metrics["rmse_mean"],
            "spearman_mean": metrics["spearman_mean"],
        })

pl.DataFrame(rows).sort("r2_mean", descending=True)

config,model,r2_mean,rmse_mean,spearman_mean
str,str,f64,f64,f64
"""engineered_rates""","""random_forest""",0.87509,5.534152,0.93219
"""compact_30""","""random_forest""",0.832163,6.484381,0.918187
"""calibrated_reduction""","""random_forest""",0.827699,6.555838,0.914199
"""multiscale_access""","""random_forest""",0.812971,6.846221,0.914848
"""domain_proxy""","""random_forest""",0.792476,7.110848,0.904822
"""calibrated_reduction""","""ridge""",0.745978,7.827951,0.864363
"""engineered_rates""","""ridge""",0.735848,7.871315,0.878844
"""multiscale_access""","""ridge""",0.720865,8.231726,0.858443
"""compact_30""","""ridge""",0.713945,8.265415,0.850607


In [37]:
for name, result in results.items():
    gi = pl.DataFrame(result["random_forest"]["group_importance"])
    print(f"\n{name} (RF r2={result['random_forest']['r2_mean']:.4f})")
    print(gi.sort("total_importance", descending=True))


domain_proxy (RF r2=0.7925)
shape: (6, 4)
┌────────────────────┬──────────────────┬─────────────────┬────────────┐
│ group              ┆ total_importance ┆ mean_importance ┆ n_features │
│ ---                ┆ ---              ┆ ---             ┆ ---        │
│ str                ┆ f64              ┆ f64             ┆ i64        │
╞════════════════════╪══════════════════╪═════════════════╪════════════╡
│ income_employment  ┆ 0.647692         ┆ 0.323846        ┆ 2          │
│ demographics       ┆ 0.178012         ┆ 0.044503        ┆ 4          │
│ barriers_housing   ┆ 0.086537         ┆ 0.017307        ┆ 5          │
│ education_health   ┆ 0.03246          ┆ 0.008115        ┆ 4          │
│ crime              ┆ 0.02998          ┆ 0.009993        ┆ 3          │
│ living_environment ┆ 0.025319         ┆ 0.00844         ┆ 3          │
└────────────────────┴──────────────────┴─────────────────┴────────────┘

compact_30 (RF r2=0.8322)
shape: (6, 4)
┌───────────────┬──────────────────┬────

In [38]:
reduced_configs = {
    "domain_proxy": domain_proxy,
    "calibrated_reduction": calibrated_reduction,
    "multiscale_access": multiscale_access,
}

for name, config in reduced_configs.items():
    df = pl.read_parquet(paths.output / f"{config.output_name}.parquet")
    _, metadata = create_feature_set(
        paths.input_file if name != "engineered_rates" else engineered_path,
        config,
    )
    print(f"\n{name}")
    print(group_summary(config, metadata))


domain_proxy
shape: (6, 6)
┌───────────────────┬────────────────┬─────────────────┬────────┬─────────────────┬────────────────┐
│ group             ┆ input_features ┆ output_features ┆ scaled ┆ reduction       ┆ mean_noise_var │
│ ---               ┆ ---            ┆ ---             ┆ ---    ┆ ---             ┆ ---            │
│ str               ┆ i64            ┆ i64             ┆ bool   ┆ str             ┆ f64            │
╞═══════════════════╪════════════════╪═════════════════╪════════╪═════════════════╪════════════════╡
│ income_employment ┆ 10             ┆ 2               ┆ true   ┆ factor_analysis ┆ 0.0            │
│ crime             ┆ 14             ┆ 3               ┆ true   ┆ factor_analysis ┆ 0.1957         │
│ barriers_housing  ┆ 20             ┆ 5               ┆ true   ┆ factor_analysis ┆ 0.3228         │
│ living_environmen ┆ 11             ┆ 3               ┆ true   ┆ factor_analysis ┆ 0.5862         │
│ t                 ┆                ┆                 ┆       

In [39]:
best_name = pl.DataFrame(rows).filter(
    pl.col("model") == "random_forest"
).sort("r2_mean", descending=True)["config"][0]

fi = pl.DataFrame(results[best_name]["random_forest"]["feature_importance"])
fi.sort("importance_mean", descending=True).head(20)

feature,group,importance_mean,importance_std
str,str,f64,f64
"""uc_claim_rate""","""benefit_rates""",0.412178,0.020902
"""uc_nwr_rate""","""benefit_rates""",0.335516,0.011429
"""youth_share""","""demographics""",0.044716,0.013156
"""transactions_per_capita""","""housing""",0.031763,0.024039
"""lsoa_max_price""","""housing""",0.028554,0.013613
…,…,…,…
"""streetlit_percentage""","""environment""",0.003782,0.001347
"""crime_rate_per_1000""","""crime_rates""",0.00368,0.000901
"""resolution_rate""","""crime_rates""",0.003578,0.00101


In [40]:
er_fi = pl.DataFrame(results["engineered_rates"]["random_forest"]["feature_importance"])
er_fi.filter(
    pl.col("group").is_in(["crime_rates", "benefit_rates", "demographics"])
).sort("importance_mean", descending=True)

feature,group,importance_mean,importance_std
str,str,f64,f64
"""uc_claim_rate""","""benefit_rates""",0.412178,0.020902
"""uc_nwr_rate""","""benefit_rates""",0.335516,0.011429
"""youth_share""","""demographics""",0.044716,0.013156
"""violent_crime_rate""","""crime_rates""",0.01753,0.004359
"""working_age_population""","""demographics""",0.00851,0.002041
…,…,…,…
"""elderly_share""","""demographics""",0.003288,0.001078
"""lsoa_population""","""demographics""",0.003003,0.000274
"""burglary_rate""","""crime_rates""",0.00287,0.000467
